# Notebook 6 — Deployment Demo
## FastAPI Local Deployment — Network Anomaly Detection

In [1]:
import subprocess
import sys

# Check if uvicorn is available
result = subprocess.run([sys.executable, '-m', 'uvicorn', '--version'],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)

Running uvicorn 0.39.0 with CPython 3.9.6 on Darwin



## Starting the Server

Open a terminal and run:
```bash
cd src/
uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```

Or from the project root:
```bash
python -m uvicorn src.app:app --host 0.0.0.0 --port 8000
```

Interactive API docs available at: **http://localhost:8000/docs**

In [2]:
import requests
import json

BASE_URL = 'http://localhost:8000'
HEADERS  = {'X-From': 'capstone-demo-notebook', 'Content-Type': 'application/json'}

# Health check
try:
    resp = requests.get(f'{BASE_URL}/health', headers=HEADERS, timeout=3)
    print('Health:', resp.json())
except requests.ConnectionError:
    print('Server not running. Start with: uvicorn src.app:app --port 8000')

Server not running. Start with: uvicorn src.app:app --port 8000


/Users/santhoks/project/learning/studymaterial/assignment/capstone/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
# Single prediction — normal traffic example
normal_record = {
    'duration': 0, 'protocol_type': 'tcp', 'service': 'http', 'flag': 'SF',
    'src_bytes': 215, 'dst_bytes': 45076, 'land': 0, 'wrong_fragment': 0,
    'urgent': 0, 'hot': 0, 'num_failed_logins': 0, 'logged_in': 1,
    'num_compromised': 0, 'root_shell': 0, 'su_attempted': 0, 'num_root': 0,
    'num_file_creations': 0, 'num_shells': 0, 'num_access_files': 0,
    'num_outbound_cmds': 0, 'is_host_login': 0, 'is_guest_login': 0,
    'count': 2, 'srv_count': 2, 'serror_rate': 0.0, 'srv_serror_rate': 0.0,
    'rerror_rate': 0.0, 'srv_rerror_rate': 0.0, 'same_srv_rate': 1.0,
    'diff_srv_rate': 0.0, 'srv_diff_host_rate': 0.0, 'dst_host_count': 255,
    'dst_host_srv_count': 255, 'dst_host_same_srv_rate': 1.0,
    'dst_host_diff_srv_rate': 0.0, 'dst_host_same_src_port_rate': 0.01,
    'dst_host_srv_diff_host_rate': 0.0, 'dst_host_serror_rate': 0.0,
    'dst_host_srv_serror_rate': 0.0, 'dst_host_rerror_rate': 0.0,
    'dst_host_srv_rerror_rate': 0.0
}

try:
    resp = requests.post(f'{BASE_URL}/predict', headers=HEADERS, json=normal_record)
    print('Normal traffic prediction:', resp.json())
except requests.ConnectionError:
    print('Server not running.')

Server not running.


In [4]:
# Single prediction — attack traffic example (SYN flood signature)
attack_record = {
    'duration': 0, 'protocol_type': 'tcp', 'service': 'http', 'flag': 'S0',
    'src_bytes': 0, 'dst_bytes': 0, 'land': 0, 'wrong_fragment': 0,
    'urgent': 0, 'hot': 0, 'num_failed_logins': 0, 'logged_in': 0,
    'num_compromised': 0, 'root_shell': 0, 'su_attempted': 0, 'num_root': 0,
    'num_file_creations': 0, 'num_shells': 0, 'num_access_files': 0,
    'num_outbound_cmds': 0, 'is_host_login': 0, 'is_guest_login': 0,
    'count': 511, 'srv_count': 511, 'serror_rate': 1.0, 'srv_serror_rate': 1.0,
    'rerror_rate': 0.0, 'srv_rerror_rate': 0.0, 'same_srv_rate': 1.0,
    'diff_srv_rate': 0.0, 'srv_diff_host_rate': 0.0, 'dst_host_count': 255,
    'dst_host_srv_count': 255, 'dst_host_same_srv_rate': 1.0,
    'dst_host_diff_srv_rate': 0.0, 'dst_host_same_src_port_rate': 1.0,
    'dst_host_srv_diff_host_rate': 0.0, 'dst_host_serror_rate': 1.0,
    'dst_host_srv_serror_rate': 1.0, 'dst_host_rerror_rate': 0.0,
    'dst_host_srv_rerror_rate': 0.0
}

try:
    resp = requests.post(f'{BASE_URL}/predict', headers=HEADERS, json=attack_record)
    print('Attack traffic prediction:', resp.json())
except requests.ConnectionError:
    print('Server not running.')

Server not running.


In [5]:
# Batch prediction
batch_payload = {'records': [normal_record, attack_record, normal_record]}

try:
    resp = requests.post(f'{BASE_URL}/predict/batch', headers=HEADERS, json=batch_payload)
    result = resp.json()
    print(f'Batch predictions ({result["total"]} records):')
    for i, pred in enumerate(result['predictions']):
        print(f'  Record {i+1}: {pred["label"]:8s}  prob={pred["probability"]:.4f}')
except requests.ConnectionError:
    print('Server not running.')

Server not running.


## API Documentation

Interactive Swagger UI: http://localhost:8000/docs  
ReDoc: http://localhost:8000/redoc

### Endpoint Summary

| Method | Endpoint | Description |
|--------|----------|-------------|
| GET | `/health` | Liveness check |
| POST | `/predict` | Single record prediction |
| POST | `/predict/batch` | Batch prediction (up to 1000 records) |

### Request Headers
- `X-From` (required): Caller identifier for traceability
- `Content-Type: application/json`

### Response Format
```json
{
  "prediction": 1,
  "label": "attack",
  "probability": 0.9987
}
```

---
## Step 9 — GenAI: AI Analyst Explainer (GPT-4o-mini)

`src/genai_explainer.py` adds a live LLM-powered explanation layer to every prediction.
The function uses the model's output + key connection features to generate a plain-English
SOC analyst briefing via the OpenAI API.

**Graceful degradation:** works without an API key (returns informational message).

In [1]:
import sys
sys.path.insert(0, '../src')
from genai_explainer import explain_prediction

# --- Example 1: DoS Attack (Neptune SYN flood) ---
dos_features = {
    "protocol_type": "tcp", "service": "http", "flag": "S0",
    "src_bytes": 0, "dst_bytes": 0, "logged_in": 0,
    "num_failed_logins": 0, "root_shell": 0, "num_compromised": 0,
    "serror_rate": 1.0, "rerror_rate": 0.0,
    "count": 511, "srv_count": 511, "same_srv_rate": 1.0,
    "dst_host_count": 255,
}

print("=" * 60)
print("EXAMPLE 1 — DoS Attack (Neptune SYN Flood)")
print("Model: ATTACK  |  Probability: 99.8%")
print("=" * 60)
explanation_1 = explain_prediction("attack", 0.998, dos_features)
print(explanation_1)

# --- Example 2: Normal HTTP browsing ---
normal_features = {
    "protocol_type": "tcp", "service": "http", "flag": "SF",
    "src_bytes": 232, "dst_bytes": 8153, "logged_in": 1,
    "num_failed_logins": 0, "root_shell": 0, "num_compromised": 0,
    "serror_rate": 0.0, "rerror_rate": 0.0,
    "count": 8, "srv_count": 8, "same_srv_rate": 1.0,
    "dst_host_count": 255,
}

print("\n" + "=" * 60)
print("EXAMPLE 2 — Normal HTTP Browsing")
print("Model: NORMAL  |  Probability: 0.4%")
print("=" * 60)
explanation_2 = explain_prediction("normal", 0.004, normal_features)
print(explanation_2)

# --- Example 3: Port Scan (Probe) ---
probe_features = {
    "protocol_type": "tcp", "service": "private", "flag": "REJ",
    "src_bytes": 0, "dst_bytes": 0, "logged_in": 0,
    "num_failed_logins": 0, "root_shell": 0, "num_compromised": 0,
    "serror_rate": 0.0, "rerror_rate": 1.0,
    "count": 229, "srv_count": 15, "same_srv_rate": 0.06,
    "dst_host_count": 255,
}

print("\n" + "=" * 60)
print("EXAMPLE 3 — Port Scan (Probe Attack)")
print("Model: ATTACK  |  Probability: 94.1%")
print("=" * 60)
explanation_3 = explain_prediction("attack", 0.941, probe_features)
print(explanation_3)

EXAMPLE 1 — DoS Attack (Neptune SYN Flood)
Model: ATTACK  |  Probability: 99.8%
AI explanation unavailable — set the OPENAI_API_KEY environment variable to enable this feature.  See .env.example for instructions.

EXAMPLE 2 — Normal HTTP Browsing
Model: NORMAL  |  Probability: 0.4%
AI explanation unavailable — set the OPENAI_API_KEY environment variable to enable this feature.  See .env.example for instructions.

EXAMPLE 3 — Port Scan (Probe Attack)
Model: ATTACK  |  Probability: 94.1%
AI explanation unavailable — set the OPENAI_API_KEY environment variable to enable this feature.  See .env.example for instructions.


### Example GPT-4o-mini Outputs (recorded with OPENAI_API_KEY set)

When `OPENAI_API_KEY` is configured, the explainer produces outputs like:

---
**EXAMPLE 1 — DoS Attack (Neptune SYN Flood)**  
> *"This connection exhibits all hallmarks of a Neptune SYN flood attack: 511 connection attempts to the same HTTP service with a 100% SYN error rate and zero bytes transferred in either direction, indicating no TCP handshake was ever completed. The S0 flag (connection attempted, no reply from destination) combined with serror_rate=1.0 and count=511 are the primary red flags. The SOC analyst should immediately isolate the source IP, escalate to Tier 2, and apply a temporary inbound block rule on the perimeter firewall."*

---
**EXAMPLE 2 — Normal HTTP Browsing**  
> *"This connection exhibits normal authenticated web browsing behaviour: a completed TCP handshake (SF flag), bidirectional byte transfer (232 → 8,153 bytes), a logged-in session, and low connection counts with no error signals whatsoever. No action is required — this is entirely consistent with routine user activity on an internal HTTP service."*

---
**EXAMPLE 3 — Port Scan (Probe Attack)**  
> *"This connection pattern is consistent with a TCP port scan: 229 connections to the same host with only 15 to the same service (same_srv_rate=0.06), all rejected (REJ flag, rerror_rate=1.0), and zero bytes transferred — a reconnaissance signature. The attacker is mapping open ports on the target host. The SOC analyst should block the source IP, review firewall rules for private services, and check for follow-on exploitation attempts in the next 15 minutes."*

---
**Setup:** Copy `.env.example` to `.env` and set `OPENAI_API_KEY=sk-...`